# Notebook 09 — window information and nonlinear benchmark

**Question:** do weak results reflect repeated window inputs, implementation problems or limitations
of the current model family? This is exploratory development analysis, not a search for a positive
result. All 32 final-test participants stay sealed. We reuse corrected immutable inputs and frozen
participant folds. Read the [frozen specification](../reports/window_benchmark_specification.md).

The Morning information audit is retrospective: the current wake answer defines only the interval.
The separate model benchmark uses the existing 60-minute cohort across all survey categories.
Only aggregates and clearly synthetic examples are displayed; source notebook outputs stay empty.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from garmin_preprocessing.common import verify_bundle  # noqa: E402
from garmin_window_benchmark.information import density  # noqa: E402

BUNDLE = ROOT / "private/window_benchmark/runs/22397f30f1439325"
REPORT = ROOT / "reports/window_benchmark/e3e34a2ea014f38f"
manifest = verify_bundle(BUNDLE)
verify_bundle(REPORT)
assert manifest["development_only"] and not manifest["final_test_access"]
print("Verified immutable development bundle; final-test data are not loaded.")

def aggregate(name):
    return pd.read_json(REPORT / (name + ".json"))


## 1. A different left boundary need not add information

Synthetic example: wake at 07:00, cutoff 08:09; accepted readings occur only at 08:05, 08:06 and
08:08. Wake, 60-minute and 5-minute windows contain the same three readings. Their latest value,
mean, SD and slope are identical even though their observation intervals differ. This does not mean
that the entire wake interval was recorded. All intervals are left-inclusive and right-exclusive.

The source audit compares original source hash/row identities, not just equal HR values. Legitimate
repeated values at different timestamps remain distinct. Near-identical vectors use prespecified
descriptive tolerances (1 bpm mean/SD/latest; .1 bpm/min slope), not new eligibility standards.

In [ ]:
overlap = aggregate("window_overlap")
primary = overlap[(overlap.scenario == "reported_minute_start") & (overlap.cohort == "pair_fixed_60")]
display(primary[["window_a", "window_b", "responses", "complete_vectors", "identical_sets", "identical_latest_identity", "near_identical_vector"]])
display(Image(filename=str(REPORT / "window_information.png")))
differences = aggregate("feature_differences")
display(differences[(differences.scenario == "reported_minute_start") & (differences.cohort == "pair_fixed_60") & (differences.window_a == "wake") & (differences.window_b == "fixed_60")][["feature", "responses", "median_absolute_difference", "q95_absolute_difference", "pearson", "spearman"]])

Wake and 60 minutes reuse exactly the same accepted source set for **116/227** Morning responses;
they share the latest reading for **227/227**. Wake and 120 minutes match for 173/227. Five-minute
windows more often differ and sometimes lack complete features. The all-window cohort (191 responses)
and later-minute sensitivity are retained in the aggregate files. These views are not independent
replications. Source overlap explains some similarity, not all weak prediction results.

## 2. Reading weights versus observed-minute weights — synthetic only

Synthetic readings: three values of 60 in minute 0, one 120 in minute 1, nothing in minute 2, and
one 90 in minute 3. The reading mean is 78; the equally weighted mean of the three **observed**
minute means is 90. The missing minute contributes neither zero nor an invented value.
This alternative is descriptive only and never enters the benchmark. It is not continuous time
weighting. Bins start at each window's own left edge; a partial final bin can be observed.

In [ ]:
synthetic_readings = pd.DataFrame({"n_epoch_s": [0., 1., 2., 60., 180.], "n_value": [60., 60., 60., 120., 90.]})
example = density(synthetic_readings, 0, 181)
assert example["reading_mean"] == 78 and example["observed_minute_mean"] == 90
print({"synthetic_only": True, "reading_mean": example["reading_mean"], "observed_minute_mean": example["observed_minute_mean"], "occupied_minutes": example["occupied_minutes"], "possible_minutes": example["possible_minutes"]})
display(Image(filename=str(REPORT / "density_weighting.png")))

## 3. Trace the implementation independently

The deterministic source trace samples two distinct participants per frozen fold, chosen by stable
EMA-key hash without using outcomes. It verifies original raw values/rows, mapping provenance,
clock offsets/buffers, boundaries and scalar features for all four windows and both wake scenarios.
No identities or full rows are displayed. The synthetic positive control exercises actual clock
parsing, acceptance, features, shuffled strict linkage and held-out prediction. Boundary decoys must
not enter features. Synthetic success verifies numerical plumbing, not physiological validity.

In [ ]:
for name in ["source_verification", "reference_agreement", "synthetic_positive_control"]:
    print(name, json.loads((REPORT / (name + ".json")).read_text()))

## 4. The nested benchmark was specified before fitting

Use unchanged 60-minute complete HR/outcome/context cases across all survey categories. Secondary
requires supported steps; refit **every** comparator on its identical cohort. Inputs are local-time
sin/cos, category and HR mean/SD/latest/slope, with steps in the secondary addition. Quality remains
separate. No imputation, answers, HRV or Garmin stress enter the models.

Five frozen outer participant folds measure OOF error. Three participant-grouped inner folds within
each training set select by mean inner MAE. Scalers and encoders fit inside each training split.
Models: matched mean/median dummies, fixed Ridge alpha=1, Ridge grid .1/1/10/100, and absolute-error
HistGradientBoosting with depth 1/2 and 50/150 iterations. Its learning rate .05, minimum leaf 20,
L2=1 and early_stopping=False are fixed. No random row-level holdout is used. Differences include
loss, regularization and model family; they cannot be attributed solely to nonlinearity.

In [ ]:
metrics = aggregate("metrics")
pooled = metrics[(metrics.stage == "oof") & (metrics.fold == "pooled") & (metrics.survey_category == "All")]
display(pooled[["block", "model", "responses", "participants", "mae", "rmse", "participant_mae", "prediction_sd"]])
display(aggregate("fold_counts"))
display(Image(filename=str(REPORT / "benchmark_errors.png")))

## 5. Performance, sample selection and uncertainty belong together

Primary: 1,634 responses / 118 development participants. Secondary: 1,355 / 113; differences across
those cohorts are not activity effects. Morning recalls since waking; Midday/Endday recall since
the last prompt. Sixty minutes is not claimed to match all those intervals.

Primary HGB context+HR MAE **2.0814** versus matching median **2.0710**: difference **+.0104**, conditional
95% interval **[−.0160,+.0392]**. Versus HGB context: **+.0073 [−.0199,+.0362]**. Secondary HGB steps
adds **+.0008 [−.0039,+.0059]** relative to matched HGB context+HR. No clear wearable advantage.

Intervals resample participant clusters with all their fixed OOF predictions, using 2,000 paired
draws. They omit refitting/tuning uncertainty and multiplicity correction. Nested tuning does not
make repeated development exploration an independent confirmation. Do not select a winning window.

In [ ]:
contrasts = aggregate("paired_intervals")
display(contrasts[contrasts.model.str.startswith("hgb__") & contrasts.baseline.isin(["median", "hgb__context", "hgb__context_hr"])])
display(aggregate("composition"))
settings = aggregate("selected_settings")
display(settings.groupby(["block", "model", "setting"]).size().rename("folds_selected").reset_index())
display(Image(filename=str(REPORT / "training_vs_oof.png")))

## 6. What we learned and the next justified step

Many Morning windows reuse readings. Independently traced features and a synthetic control support
implementation correctness. The flexible HGB model produces varying predictions, but lower training
error does not establish OOF improvement beyond matched median/context baselines. Tuned Ridge does
not change that conclusion. These checks do not prove that the wearable has no physiological signal.

Recommend provider timing/quality evidence before expanding model searches. Draft questions:

- Does `ts` locate acquisition or receipt, and how do batching, ordering and duplicate exports work?
- Are actual notification/delivery/display logs available to identify the previous prompt?
- Which device wear/contact, motion, confidence and beat-quality metadata exist, and what intervals
  do step-counter updates represent?

These questions require provider knowledge and have **not** been sent. We can test alternate
observed-minute weighting ourselves in a separately frozen experiment, but this descriptive result
does not justify silently replacing predictors. Keep the final test sealed.

In [ ]:
print("Report:", REPORT.relative_to(ROOT) / "report.md")
print("Professor summary:", REPORT.relative_to(ROOT) / "professor_summary.md")
for name in ["window_pairs.parquet", "window_density.parquet", "source_trace.parquet", "membership.parquet", "training_predictions.parquet", "out_of_fold_predictions.parquet"]:
    print("Ignored diagnostic output:", BUNDLE.relative_to(ROOT) / name)
print("No commit, push, external messages or final-test evaluation.")